# 1. IMPORTAÇÕES


In [0]:

# Análise e ingestão
import pandas as pd

# Spark
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col,
    lit,
    current_timestamp,
    count,
    when
)

# 2. URLS


In [0]:
URL_CDI = "https://raw.githubusercontent.com/miriamstal26-rgb/mvp-financiamento-cdi-ipca/refs/heads/main/data/landing/cdi_4389.csv"

URL_SELIC = "https://raw.githubusercontent.com/miriamstal26-rgb/mvp-financiamento-cdi-ipca/refs/heads/main/data/landing/selic_432.csv"

URL_IPCA = "https://raw.githubusercontent.com/miriamstal26-rgb/mvp-financiamento-cdi-ipca/refs/heads/main/data/landing/ipca_433.csv"

# 3. INGESTÃO DOS DADOS

In [0]:
# INGESTÃO DO CDI

df_cdi = pd.read_csv(
    URL_CDI,
    sep=";"
)


In [0]:
# INGESTÃO DA SELIC

df_selic = pd.read_csv(
    URL_SELIC,
    sep=";"
)


In [0]:
# INGESTÃO DO IPCA

df_ipca = pd.read_csv(
    URL_IPCA,
    sep=";"
)

# 4. VALIDAÇÃO INICIAL

In [0]:
# CDI 

df_cdi.head()
df_cdi.tail()
df_cdi.info()
df_cdi.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2509 entries, 0 to 2508
Data columns (total 2 columns):
 #   Column                                                   Non-Null Count  Dtype 
---  ------                                                   --------------  ----- 
 0   Data                                                     2509 non-null   object
 1   4389 - Taxa de juros - CDI anualizada base 252 - % a.a.  2509 non-null   object
dtypes: object(2)
memory usage: 39.3+ KB


Data                                                       0
4389 - Taxa de juros - CDI anualizada base 252 - % a.a.    0
dtype: int64

In [0]:
# IPCA

df_ipca.head()
df_ipca.tail()
df_ipca.info()
df_ipca.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 2 columns):
 #   Column                                                                      Non-Null Count  Dtype 
---  ------                                                                      --------------  ----- 
 0   Data                                                                        121 non-null    object
 1   433 - Índice nacional de preços ao consumidor-amplo (IPCA) - Var. % mensal  121 non-null    object
dtypes: object(2)
memory usage: 2.0+ KB


Data                                                                          0
433 - Índice nacional de preços ao consumidor-amplo (IPCA) - Var. % mensal    0
dtype: int64

In [0]:
# SELIC

df_selic.head()
df_selic.tail()
df_selic.info()
df_selic.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3653 entries, 0 to 3652
Data columns (total 2 columns):
 #   Column                                                         Non-Null Count  Dtype 
---  ------                                                         --------------  ----- 
 0   Data                                                           3653 non-null   object
 1   432 - Taxa de juros - Meta Selic definida pelo Copom - % a.a.  3653 non-null   object
dtypes: object(2)
memory usage: 57.2+ KB


Data                                                             0
432 - Taxa de juros - Meta Selic definida pelo Copom - % a.a.    0
dtype: int64

# 5. TRATAMENTO DE REGISTROS NÃO PERTENCENTES À SÉRIE

Durante a inspeção inicial dos dados por meio da função `tail()`, foi identificado que os arquivos exportados pelo Sistema Gerenciador de Séries Temporais (SGS) apresentam um registro adicional ao final da série contendo informações sobre a fonte dos dados.

Exemplos identificados:

- CDI → Fonte = BCB-Demab
- Selic → Fonte = Copom
- IPCA → Fonte = IBGE

Embora essas informações sejam relevantes para fins de documentação e rastreabilidade, elas não representam observações válidas da série temporal analisada. Sua permanência nas tabelas poderia comprometer a consistência estrutural dos dados, especialmente em etapas posteriores de tratamento, conversão de tipos e análises temporais.

Dessa forma, foi implementada uma função reutilizável para remover esses registros de metadados das três bases, preservando apenas as observações efetivamente pertencentes às séries históricas.

Esta transformação foi realizada ainda na camada Bronze por se tratar de uma correção estrutural da fonte de dados, e não de uma transformação analítica ou de negócio.

In [0]:
df_cdi.tail(3)

,Data,4389 - Taxa de juros - CDI anualizada base 252 - % a.a.
2506,29/06/2026,"14,15"
2507,30/06/2026,"14,15"
2508,Fonte,BCB-Demab


In [0]:
df_ipca.tail(3)

,Data,433 - Índice nacional de preços ao consumidor-amplo (IPCA) - Var. % mensal
118,05/2026,"0,58"
119,06/2026,"0,16"
120,Fonte,IBGE


In [0]:
df_selic.tail(3)

,Data,432 - Taxa de juros - Meta Selic definida pelo Copom - % a.a.
3650,29/06/2026,"14,25"
3651,30/06/2026,"14,25"
3652,Fonte,Copom


In [0]:
# FUNÇÃO PARA REMOÇÃO DE REGISTROS DE METADADOS

def remover_linha_fonte(df):
    """
    Remove a linha de metadado 'Fonte'
    presente ao final dos arquivos exportados pelo SGS.
    """
    return df[df.iloc[:, 0] != "Fonte"]

In [0]:
# Aplicando tratamento

df_cdi = remover_linha_fonte(df_cdi)
df_selic = remover_linha_fonte(df_selic)
df_ipca = remover_linha_fonte(df_ipca)

In [0]:
df_cdi.tail(3)

,Data,4389 - Taxa de juros - CDI anualizada base 252 - % a.a.
2505,26/06/2026,"14,15"
2506,29/06/2026,"14,15"
2507,30/06/2026,"14,15"


In [0]:
df_selic.tail(3)


,Data,432 - Taxa de juros - Meta Selic definida pelo Copom - % a.a.
3649,28/06/2026,"14,25"
3650,29/06/2026,"14,25"
3651,30/06/2026,"14,25"


In [0]:
df_ipca.tail(3)

,Data,433 - Índice nacional de preços ao consumidor-amplo (IPCA) - Var. % mensal
117,04/2026,"0,67"
118,05/2026,"0,58"
119,06/2026,"0,16"


# 6. PADRONIZAÇÃO ESTRUTURAL DAS COLUNAS

Após a remoção dos registros de metadados identificados na etapa anterior, foi realizada a padronização dos nomes das colunas das três séries econômicas.

Os arquivos exportados pelo Sistema Gerenciador de Séries Temporais (SGS) apresentam nomes de colunas extensos e específicos para cada indicador, contendo descrições completas das séries. Embora essas informações sejam úteis para consulta humana, elas dificultam a padronização do pipeline e a reutilização das transformações entre diferentes conjuntos de dados.

Dessa forma, os nomes das colunas foram padronizados para os atributos genéricos `data` e `valor`, criando uma estrutura comum entre as séries de CDI, Selic e IPCA. Essa alteração não modifica o conteúdo dos dados, apenas sua representação estrutural, facilitando as etapas posteriores de transformação e integração dos dados.

In [0]:
df_cdi.columns = ["data", "valor"]

df_selic.columns = ["data", "valor"]

df_ipca.columns = ["data", "valor"]

# 7. INCLUSÃO DE METADADOS DE INGESTÃO

Como parte das boas práticas de Engenharia de Dados, foram adicionados metadados de controle e rastreabilidade aos registros da camada Bronze.

Foram incluídos os atributos:

- `data_ingestao`: registra a data e hora em que o dado foi processado pelo pipeline;
- `arquivo_origem`: identifica o arquivo de origem responsável pela geração do registro.

Esses metadados possibilitam auditoria, monitoramento e rastreamento da origem dos dados ao longo das demais camadas da arquitetura. Sua utilização é comum em ambientes Lakehouse e auxilia na governança dos dados, permitindo identificar quando e de onde cada informação foi carregada.

In [0]:
from datetime import datetime

momento_ingestao = datetime.now()

df_cdi["data_ingestao"] = momento_ingestao
df_cdi["arquivo_origem"] = "cdi_4389.csv"

df_selic["data_ingestao"] = momento_ingestao
df_selic["arquivo_origem"] = "selic_432.csv"

df_ipca["data_ingestao"] = momento_ingestao
df_ipca["arquivo_origem"] = "ipca_433.csv"

In [0]:
df_cdi.head()

,data,valor,data_ingestao,arquivo_origem
0,01/07/2016,"14,13",2026-09-17 19:37:28.756909,cdi_4389.csv
1,04/07/2016,"14,13",2026-09-17 19:37:28.756909,cdi_4389.csv
2,05/07/2016,"14,13",2026-09-17 19:37:28.756909,cdi_4389.csv
3,06/07/2016,"14,13",2026-09-17 19:37:28.756909,cdi_4389.csv
4,07/07/2016,"14,13",2026-09-17 19:37:28.756909,cdi_4389.csv


# 8. CONVERSÃO PARA DATAFRAMES SPARK

Após a etapa de ingestão e tratamento inicial utilizando Pandas, os dados são convertidos para DataFrames Spark para permitir o armazenamento e processamento distribuído dentro do ambiente Databricks.

Essa conversão marca a transição dos dados da etapa de preparação local para a estrutura de processamento utilizada pelo Lakehouse.

In [0]:
spark_cdi = spark.createDataFrame(df_cdi)

spark_selic = spark.createDataFrame(df_selic)

spark_ipca = spark.createDataFrame(df_ipca)

In [0]:
spark_cdi.printSchema()

spark_selic.printSchema()

spark_ipca.printSchema()

root
 |-- data: string (nullable = true)
 |-- valor: string (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)
 |-- arquivo_origem: string (nullable = true)

root
 |-- data: string (nullable = true)
 |-- valor: string (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)
 |-- arquivo_origem: string (nullable = true)

root
 |-- data: string (nullable = true)
 |-- valor: string (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)
 |-- arquivo_origem: string (nullable = true)



In [0]:
display(spark_cdi)

data,valor,data_ingestao,arquivo_origem
01/07/2016,"14,13",2026-09-17T19:37:28.756Z,cdi_4389.csv
04/07/2016,"14,13",2026-09-17T19:37:28.756Z,cdi_4389.csv
05/07/2016,"14,13",2026-09-17T19:37:28.756Z,cdi_4389.csv
06/07/2016,"14,13",2026-09-17T19:37:28.756Z,cdi_4389.csv
07/07/2016,"14,13",2026-09-17T19:37:28.756Z,cdi_4389.csv
08/07/2016,"14,13",2026-09-17T19:37:28.756Z,cdi_4389.csv
11/07/2016,"14,13",2026-09-17T19:37:28.756Z,cdi_4389.csv
12/07/2016,"14,13",2026-09-17T19:37:28.756Z,cdi_4389.csv
13/07/2016,"14,13",2026-09-17T19:37:28.756Z,cdi_4389.csv
14/07/2016,"14,13",2026-09-17T19:37:28.756Z,cdi_4389.csv


# 9. PERSISTÊNCIA DA CAMADA BRONZE

Após a ingestão e preparação inicial dos dados, os DataFrames Spark foram persistidos como tabelas da camada Bronze.

O objetivo dessa etapa é disponibilizar os dados de forma estruturada dentro do ambiente Lakehouse, preservando sua granularidade original e mantendo os metadados de rastreabilidade adicionados durante a ingestão.

As tabelas geradas nesta etapa servirão como fonte para os processos de transformação da camada Silver.

In [0]:
spark_cdi.write \
    .mode("overwrite") \
    .saveAsTable("bronze_cdi")

In [0]:
spark_selic.write \
    .mode("overwrite") \
    .saveAsTable("bronze_selic")

In [0]:
spark_ipca.write \
    .mode("overwrite") \
    .saveAsTable("bronze_ipca")

In [0]:
spark.sql("SHOW TABLES").show(truncate=False)

+--------+------------+-----------+
|database|tableName   |isTemporary|
+--------+------------+-----------+
|default |bronze_cdi  |false      |
|default |bronze_ipca |false      |
|default |bronze_selic|false      |
+--------+------------+-----------+



%md
# Conclusão da Camada Bronze

A camada Bronze foi implementada com sucesso.

Resultados obtidos:

- Ingestão dos dados da Landing Zone.
- Validação estrutural das séries.
- Remoção de registros de metadados.
- Padronização estrutural das colunas.
- Inclusão de metadados de ingestão.
- Conversão para DataFrames Spark.
- Persistência das tabelas Bronze no Lakehouse.

Tabelas geradas:

- bronze_cdi
- bronze_selic
- bronze_ipca